### **Notebook 02** : Logistic Regression — Model Training & Experimentation
### **Input**  : X_train/test, y_train/test (from Notebook 01), scaler.pkl
### **Output** : lr_model.pkl, experiment results table

---
### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix, roc_curve
)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.titlesize'] = 13

print('Imports successful')

Imports successful


### Experiment Configurations

Central configuration for the experiment. Adjust the settings below to 
control feature selection, scaling, and model hyperparameters — then 
re-run from Section 5 onwards to see the impact.

- **`FEATURE_COLS`** — select which features to include in training
- **`SCALER_TYPE`** — normalization method (`standard` / `minmax` / `robust`)
- **`MODEL_PARAMS`** — logistic regression hyperparameters (`C`, `penalty`, `solver`)

In [5]:
# Data paths
DATA_DIR      = '../outputs/csv/'
OUT_DIR       = '../outputs/'

# Feature selections for the training
FEATURE_COLS = [
    'Recency',           # days since last purchase
    'Frequency',         # number of orders
    'Monetary',          # total spend
    'AvgOrderValue',     # average order size
    'UniqueProducts',    # variety of items bought
    'PurchaseSpanDays',  # days between first and last purchase
]

# Select scaler type | standard, minmax, robust
SCALER_TYPE = "minmax"

# Model trianing hyperparameters
MODEL_PARAMS  = dict(
    C = 0.1,       # regularisation strength — smaller = stronger reg
    penalty = 'l2',      # 'l1' | 'l2' | 'elasticnet' | None
    solver = 'saga',   # 'lbfgs' | 'saga' | 'liblinear'
    max_iter = 2000,
    random_state = 42,
)

print('Config loaded.')
print(f'  Features      : {FEATURE_COLS}')
print(f'  Scaling       : {SCALER_TYPE}')
print(f'  C / penalty   : {MODEL_PARAMS["C"]} / {MODEL_PARAMS["penalty"]}')



Config loaded.
  Features      : ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'PurchaseSpanDays']
  Scaling       : minmax
  C / penalty   : 0.1 / l2


### Load data

Loads the preprocessed train/test splits produced by Notebook 01.
Features (`X`) and labels (`y`) are loaded separately for both sets.

In [10]:
X_train = pd.read_csv(DATA_DIR + 'X_train.csv')
X_test  = pd.read_csv(DATA_DIR + 'X_test.csv')
y_train = pd.read_csv(DATA_DIR + 'y_train.csv').squeeze()
y_test  = pd.read_csv(DATA_DIR + 'y_test.csv').squeeze()

print(f'X_train : {X_train.shape}   X_test : {X_test.shape}')
print(f'Class balance (train) — 0: {(y_train==0).sum()}  1: {(y_train==1).sum()} ({y_train.mean():.1%} positive)')
X_train.head()

X_train : (2883, 6)   X_test : (721, 6)
Class balance (train) — 0: 1416  1: 1467 (50.9% positive)


,Recency,Frequency,Monetary,AvgOrderValue,UniqueProducts,PurchaseSpanDays
0,140,2,662.08,331.040000,17,70
1,53,4,2411.82,602.955000,195,154
2,8,3,534.12,178.040000,35,245
3,112,1,308.68,308.680000,18,0
4,9,3,469.00,156.333333,57,197


In [11]:
print(list(X_train.columns))

['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'PurchaseSpanDays']


### Feature transformation - Transform existing features to improve model performance

Base features were extracted from raw UCI Online Retail transactional data 
as per-customer behavioural summaries.

Additional features are constructed via **feature transformation** to improve 
model performance:
- **Log transforms** : reduce skewness in tailed distributions
- **Ratio features** : capture relationships between existing features
- **Binning** : converts continuous values into ordinal categories

In [13]:
def add_engineered_features(df):
    df = df.copy()

    # Log transforms (reduce skew for LR) 
    df['LogRecency']   = np.log1p(df['Recency'])
    df['LogMonetary']  = np.log1p(df['Monetary'])
    df['LogFrequency'] = np.log1p(df['Frequency'])

    # Ratio features 
    df['FreqPerSpanDay']  = df['Frequency'] / (df['PurchaseSpanDays'] + 1)   # avoid div/0
    df['MonetaryPerFreq'] = df['Monetary']  / (df['Frequency'] + 1)

    # Recency bucket (ordinal 0–4, lower = more recent) 
    df['RecencyBucket'] = pd.cut(df['Recency'], bins=5, labels=False)

    return df

X_train = add_engineered_features(X_train)
X_test  = add_engineered_features(X_test)

print('Engineered features added. Available columns:')
print(list(X_train.columns))
print(X_train.shape, X_test.shape)

Engineered features added. Available columns:
['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'PurchaseSpanDays', 'LogRecency', 'LogMonetary', 'LogFrequency', 'FreqPerSpanDay', 'MonetaryPerFreq', 'RecencyBucket']
(2883, 12) (721, 12)


In [ ]:
# check features are available
missing_cols = [c for c in FEATURE_COLS if c not in X_train.columns]
if missing_cols:
    raise ValueError(f'These features are not in the dataset: {missing_cols}\n'
                     f'Run Section 4 first, or check for typos in FEATURE_COLS.')

Xtr = X_train[FEATURE_COLS].copy()
Xte = X_test[FEATURE_COLS].copy()

# scaling
if SCALER_TYPE:
    scalers = {
        'standard' : StandardScaler(),
        'minmax'   : MinMaxScaler(),
        'robust'   : RobustScaler(),
    }
    scaler = scalers[SCALER_TYPE]
    Xtr = pd.DataFrame(scaler.fit_transform(Xtr), columns=FEATURE_COLS)
    Xte = pd.DataFrame(scaler.transform(Xte),     columns=FEATURE_COLS)
    print(f'Scaling applied: {SCALER_TYPE}')
else:
    print('No scaling applied.')


print(f'\nFinal training shape : {Xtr.shape}')
print(f'Features used        : {FEATURE_COLS}')

Scaling applied: minmax

Final training shape : (2883, 6)
Features used        : ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'PurchaseSpanDays']
